# 14. Paired contrasts, uncertainty, and decision thresholds

![Paired inference: pair first, quantify uncertainty, then decide](../images/14_paired_inference.svg)

One idea carries this notebook: form one number per independent unit first, then do statistics on those numbers. Here the independent unit is a paired trained-model block. Participants and queries sharpen each cell mean, but they do not set the primary denominator.

**Learning goals:** preserve eight paired allocation blocks, compute the residual phase-depth versus breadth contrast, use Student $t$ intervals over blocks, evaluate the four-block jitter diagnostic, and separate materiality, superiority, equivalence, and supporting decision gates.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 14
rng = np.random.Generator(np.random.PCG64(SEED))
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")


## 1. Eight blocks carry the primary allocation path

The primary study path has three allocations: breadth, balanced, and phase depth. Every primary block contains all three trained models. That matching matters because the models in one block share nuisance streams that should cancel in a within-block subtraction.

The synthetic tensor below uses shape `(block, allocation, participant)`, with allocations ordered as breadth, balanced, and phase depth. The participant axis is averaged before inference. After that reduction, the primary contrast has eight block values, not hundreds or thousands of participant rows.


In [ ]:
B, P = 8, 308
allocations = np.array(["breadth", "balanced", "phase_depth"])
gfc_mean = np.array([0.30, 0.34, 0.41])
completion_mean = np.array([0.23, 0.24, 0.25])
block_shift = rng.normal(0, 0.018, size=(B, 1, 1))
block_allocation_noise = rng.normal(0, 0.010, size=(B, 3, 1))
participant_shift = rng.normal(0, 0.050, size=(1, 1, P))
gfc = np.clip(gfc_mean[None, :, None] + block_shift + block_allocation_noise + participant_shift + rng.normal(0, 0.030, (B, 3, P)), -1, 1)
completion = np.clip(completion_mean[None, :, None] + 0.65 * block_shift + 0.7 * participant_shift + rng.normal(0, 0.025, (B, 3, P)), -1, 1)
assert gfc.shape == completion.shape == (8, 3, 308)
gfc_cell = gfc.mean(axis=-1)
completion_cell = completion.mean(axis=-1)
assert gfc_cell.shape == completion_cell.shape == (8, 3)
print(allocations.tolist())
print("cell mean shape:", gfc_cell.shape)


## 2. Residual margins and the primary block contrast

GFC and independent completion use the same gallery and the same margin scale. Subtracting completion from GFC inside each block and allocation gives a residual score `D`. That residual asks how much donor-based recombination remains after the part explained by independent factor recovery is removed.

The confirmatory block contrast is phase depth minus breadth: $P_r=D_{r,phase}-D_{r,breadth}$. Balanced is still reported, because it shows the path shape, but it is not needed for the primary subtraction.

![One paired block holding breadth, balanced, and phase-depth models, then computing the residual phase-depth minus breadth contrast inside that block](../images/14_current_block_contrast.svg)


In [ ]:
D = gfc_cell - completion_cell
breadth = D[:, 0]
balanced = D[:, 1]
phase_depth = D[:, 2]
primary = phase_depth - breadth
path_steps = np.column_stack([balanced - breadth, phase_depth - balanced])
assert D.shape == (8, 3)
assert primary.shape == (8,)
assert np.allclose(primary, path_steps.sum(axis=1))
print("mean residuals:", dict(zip(allocations, D.mean(axis=0).round(3))))
print(f"mean primary contrast={primary.mean():.3f}")


## 3. Student $t$ intervals use eight values

Once the primary contrast exists, the analysis is a one-sample problem over eight block values. The confidence interval uses seven degrees of freedom because the standard deviation is estimated across eight blocks.

The materiality rule has two parts: the 95 percent interval excludes zero, and the point estimate reaches the declared margin. Equivalence asks a different question and uses a 90 percent interval inside the declared practical band.


In [ ]:
def t_interval(values, confidence=0.95):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(len(values))
    critical = stats.t.ppf((1 + confidence) / 2, df=len(values) - 1)
    return mean, (mean - critical * se, mean + critical * se)

def materially_positive(values, margin):
    mean, interval = t_interval(values, 0.95)
    return interval[0] > 0 and mean >= margin

def equivalent(values, margin):
    _, interval = t_interval(values, 0.90)
    return interval[0] > -margin and interval[1] < margin

margin = 0.0625
primary_summary = t_interval(primary, 0.95)
assert len(primary) == 8
print("primary mean and 95% interval:", primary_summary)
print("materially positive:", materially_positive(primary, margin))
print("equivalent at margin:", equivalent(primary, margin))


## 4. The jitter diagnostic has its own lower-precision contrast

Nearby jitter is not a fourth path point. It is a paired diagnostic for the phase-depth end of the path. In four prespecified blocks, phase depth and nearby jitter share sequence draws, base phases, masks, transforms, exposure, and seeds. The only intended difference is separated phase content versus local start variation.

The diagnostic contrast is $J_r=D_{r,phase}-D_{r,jitter}$. It has four values, so the interval is wider and the interpretation is narrower. It can support a mechanism reading only beside the primary path result.


In [ ]:
J_BLOCKS = 4
phase_residual_j = D[:J_BLOCKS, 2]
nearby_jitter = phase_residual_j - rng.normal(0.045, 0.018, size=J_BLOCKS)
jitter_contrast = phase_residual_j - nearby_jitter
jitter_summary = t_interval(jitter_contrast, 0.95)
assert jitter_contrast.shape == (4,)
assert jitter_summary[1][0] < jitter_summary[1][1]
print("jitter diagnostic mean and 95% interval:", jitter_summary)


## 5. Supporting gates do not create new primary tests

The primary residual contrast is the confirmatory test. Directional rank endpoints, the balanced path point, the jitter diagnostic, and a locked geometry diagnostic explain whether the result has the expected shape. They are supporting checks, not extra ways to select a headline.

The synthetic gate below requires a materially positive primary contrast, matching top-1 and MRR directions computed from separate rank endpoints, and a positive jitter diagnostic. A real analysis would also include the frozen geometry diagnostic. The output is a label, not a new p-value.


In [ ]:
rank_block_shift = rng.normal(0, 0.010, size=(B, 1, 1))
top1_scores = np.clip(np.array([0.56, 0.58, 0.62])[None, :, None] + rank_block_shift + rng.normal(0, 0.050, size=(B, 3, P)), 0, 1)
mrr_scores = np.clip(np.array([0.68, 0.70, 0.74])[None, :, None] + 0.5 * rank_block_shift + rng.normal(0, 0.035, size=(B, 3, P)), 0, 1)
top1_contrast = top1_scores[:, 2].mean(axis=-1) - top1_scores[:, 0].mean(axis=-1)
mrr_contrast = mrr_scores[:, 2].mean(axis=-1) - mrr_scores[:, 0].mean(axis=-1)
expected_sign = np.sign(primary.mean())
top1_direction_agrees = np.sign(top1_contrast.mean()) == expected_sign
mrr_direction_agrees = np.sign(mrr_contrast.mean()) == expected_sign
jitter_direction_agrees = jitter_contrast.mean() > 0
supporting_label = (
    materially_positive(primary, margin)
    and top1_direction_agrees
    and mrr_direction_agrees
    and jitter_direction_agrees
)
assert top1_contrast.shape == mrr_contrast.shape == (8,)
assert isinstance(supporting_label, (bool, np.bool_))
print("rank directions:", top1_contrast.mean(), mrr_contrast.mean())
print("supporting phase-depth label:", bool(supporting_label))


## 6. Plot every block value

With eight primary blocks, hiding the individual values is poor practice. The interval summarizes them, but the block dots show whether the result is broad agreement, one influential block, or a split pattern.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(1, B + 1)
ax.axhline(0, color="black", linewidth=1)
ax.axhline(margin, color="#9f3f3f", linestyle="--", label="materiality margin")
ax.scatter(x, primary, color="#2563eb", label="P_r")
mean, interval = primary_summary
ax.errorbar([B + 1], [mean], yerr=[[mean - interval[0]], [interval[1] - mean]], fmt="o", color="#d97706", label="mean and 95% CI")
ax.set(xlabel="paired block", ylabel="residual phase depth minus breadth", title="Primary block contrasts")
ax.legend()
plt.tight_layout()
plt.show()


## Exercises, limits, and takeaways

1. Change the participant count from 308 to 1000. Which quantities change, and why does the primary degrees of freedom stay at seven?
2. Compute the primary contrast directly from GFC without subtracting completion. What interpretation would that lose?
3. Resample allocation cells independently across blocks. Which covariance did you destroy?
4. Change the jitter diagnostic to eight blocks after seeing outcomes. Why would that be a protocol change rather than a sensitivity?

**Takeaway:** paired inference starts by naming the independent unit. For the active study, that unit is the paired model block. The primary contrast is residual phase depth minus breadth inside each block. Supporting checks can explain that contrast, but they cannot replace it after outcomes are known.


## Continue learning

[Previous notebook: 13](13_context_interventions.ipynb) | [Lecture](../lectures/14_paired_inference.md) | [Curriculum](../README.md) | [Next notebook: 15](15_exposure_and_replication.ipynb)
